# Анализ целевой аудитории

**Используемые источники:**
1. Wikipedia, статический парсинг по населению городов РФ
2. World Bank API, демография и доходы в РФ
3. Mall Customers, открытый датасет по сегментации покупателей
4. Supermarket sales по категории Health and beauty
6. VK API, посты в пабликах для разбора клиентских болей

## 1. Настройки и палитра

In [ ]:
from pathlib import Path
ROOT = Path.cwd()
DATA_RAW = ROOT / "data" / "raw"
DATA_PROCESSED = ROOT / "data" / "processed"
FIGURES = ROOT / "figures"

for _folder in (DATA_RAW, DATA_PROCESSED, FIGURES):
    _folder.mkdir(parents=True, exist_ok=True)
RANDOM_SEED = 42

#Города, по которым смотрим зарплаты молодых специалистов
CITIES = [
    "Москва",
    "Санкт-Петербург",
    "Новосибирск",
    "Екатеринбург",
    "Казань",
    "Краснодар",
    "Нижний Новгород",
    "Ростов-на-Дону",
]

#Палитра графиков: пастельно-оранжевые оттенки
PALETTE = ["#FF7A00", "#FF4D00", "#FFB300", "#C73E00", "#FF9100", "#E8360C", "#FFC400"]
ACCENT_COLOR = "#FF820D"

SEQ_COLORS = ["#FFD199", "#FFB347", "#FF8C00", "#FF6A00", "#E8480C", "#B23A00"]
TABLE_HEADER = "#FF6A00"
TABLE_ROW_ALT = "#FFE3C7"

GENDER_COLORS = {"Женский": "#FF6FA5", "Мужской": "#FF7A00"}

TOPICS = ["Стиль и гардероб", "Уход за кожей", "Базовый луксмаксинг", "Уход за волосами", "Уверенность в себе"]

PAIN_KEYWORDS = {
    "Не знаю с чего начать": ["с чего начать", "не знаю что", "запуталась", "запутался", "слишком много информации", "глаза разбегаются"],
    "Дорого ходить к специалистам": ["дорого", "стилист дорого", "не по карману", "цена кусается", "косметолог дорого"],
    "Стесняюсь внешности": ["стесняюсь", "комплексую", "не нравлюсь себе", "неуверенность", "стыдно"],
    "Нет времени разбираться": ["нет времени", "некогда", "времени не хватает", "загружен"],
    "Боюсь сделать хуже": ["боюсь испортить", "сделать хуже", "навредить коже", "страшно пробовать"],
    "Не понимаю что мне идёт": ["что мне идёт", "не понимаю свой тип", "не подходит", "не мой стиль"],
}

## 2. Источники данных (ООП, общий интерфейс BaseSource)

In [ ]:
from abc import ABC, abstractmethod
from pathlib import Path
import pandas as pd

class BaseSource(ABC):
    cache_name: str = "source.csv"

    def __init__(self, cache_dir: Path):
        self.cache_dir = Path(cache_dir)
        self.cache_dir.mkdir(parents=True, exist_ok=True)

    @property
    def cache_path(self) -> Path:
        return self.cache_dir / self.cache_name
    
    @abstractmethod
    def fetch(self) -> pd.DataFrame:
        raise NotImplementedError

    def load(self, refresh: bool = False) -> pd.DataFrame:
        if not refresh and self.cache_path.exists():
            return pd.read_csv(self.cache_path)
        try:
            df = self.fetch()
        except Exception as err:
            if self.cache_path.exists():
                print(f"[{self.__class__.__name__}] не удалось обновить ({err}), беру кэш")
                return pd.read_csv(self.cache_path)
            raise
        self.save(df)
        return df

    def save(self, df: pd.DataFrame) -> None:
        df.to_csv(self.cache_path, index=False)
        print(f"[{self.__class__.__name__}] сохранено строк: {len(df)} -> {self.cache_path.name}")

In [ ]:
#Статический парсинг Wikipedia: города России по населению
import re
from io import StringIO

import pandas as pd
import requests
from bs4 import BeautifulSoup

URL = (
    "https://ru.wikipedia.org/wiki/"
    "Список_городов_России_с_населением_более_100_тысяч_жителей"
)
HEADERS = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"}

class WikipediaCitiesParser(BaseSource):
    cache_name = "cities.csv"
    def fetch(self) -> pd.DataFrame:
        html = requests.get(URL, headers=HEADERS, timeout=20).text
        table = self._pick_table(html)
        return self._tidy(table)

    @staticmethod
    def _pick_table(html: str) -> pd.DataFrame:
        soup = BeautifulSoup(html, "lxml")
        wikitables = soup.select("table.wikitable")
        for tbl in wikitables:
            df = pd.read_html(StringIO(str(tbl)))[0]
            if df.shape[0] > 50:
                return df
        raise RuntimeError("не нашёл таблицу городов на странице")

    def _tidy(self, df: pd.DataFrame) -> pd.DataFrame:
        df.columns = [" ".join(str(c) for c in col) if isinstance(col, tuple) else str(col) for col in df.columns]
        city_col = next(c for c in df.columns if "Город" in c)
        year_cols = [(c, int(m.group())) for c in df.columns if (m := re.search(r"(19|20)\d{2}", c))]
        latest_col, latest_year = max(year_cols, key=lambda x: x[1])

        out = pd.DataFrame(
            {
                "city": df[city_col].astype(str).str.replace(r"\[.*?\]", "", regex=True).str.strip(),
                "population_k": pd.to_numeric(df[latest_col], errors="coerce"),
            }
        )
        out = out.dropna(subset=["population_k"])
        out = out[~out["city"].str.contains("итог|город", case=False, na=False)]
        out["population"] = (out["population_k"] * 1000).astype(int)
        out["year"] = latest_year
        return out[["city", "population", "year"]].sort_values("population", ascending=False).reset_index(drop=True)

In [ ]:
#Реальные макропоказатели России из World Bank API
import time
import pandas as pd
import requests

API = "https://api.worldbank.org/v2/country/RUS/indicator/{code}?format=json&per_page=100&date=2010:2024"

INDICATORS = {
    "Население": "SP.POP.TOTL",
    "Доля 15-64 лет, %": "SP.POP.1564.TO.ZS",
    "Интернет-пользователи, %": "IT.NET.USER.ZS",
    "ВВП на душу, $": "NY.GDP.PCAP.CD",
    "Городское население, %": "SP.URB.TOTL.IN.ZS",
}

class WorldBankSource(BaseSource):
    cache_name = "worldbank.csv"
    def fetch(self) -> pd.DataFrame:
        frames = []
        for name, code in INDICATORS.items():
            payload = self._get(API.format(code=code))
            rows = payload[1] if len(payload) > 1 and payload[1] else []
            for row in rows:
                if row["value"] is not None:
                    frames.append({"indicator": name, "year": int(row["date"]), "value": row["value"]})
        if not frames:
            raise RuntimeError("World Bank не вернул данных")
        return pd.DataFrame(frames).sort_values(["indicator", "year"]).reset_index(drop=True)

    @staticmethod
    def _get(url: str, attempts: int = 3, timeout: int = 40):
        last_err = None
        for i in range(attempts):
            try:
                return requests.get(url, timeout=timeout).json()
            except requests.RequestException as err:
                last_err = err
                time.sleep(2 * (i + 1))
        raise RuntimeError(f"World Bank недоступен после {attempts} попыток: {last_err}")

    @staticmethod
    def latest(df: pd.DataFrame) -> dict:
        out = {}
        for name, group in df.groupby("indicator"):
            row = group.sort_values("year").iloc[-1]
            out[name] = {"value": row["value"], "year": int(row["year"])}
        return out

In [ ]:
from io import StringIO
import pandas as pd
import requests

URL = "https://raw.githubusercontent.com/selva86/datasets/master/Mall_Customers_Int.csv"

class MallCustomersSource(BaseSource):

    cache_name = "mall_customers.csv"

    def fetch(self) -> pd.DataFrame:
        text = requests.get(URL, timeout=30, headers={"User-Agent": "Mozilla/5.0"}).text
        df = pd.read_csv(StringIO(text)).dropna()
        out = pd.DataFrame(
            {
                "customer_id": df["CustomerID"],
                "gender": df["Genre"].map({0: "Женский", 1: "Мужской"}),
                "age": df["Age"].astype(int),
                "annual_income_kusd": df["Annual_Income_(k$)"].astype(float),
                "spending_score": df["Spending_Score"].astype(float),
            }
        )
        return out.dropna().reset_index(drop=True)

In [ ]:
#Реальные продажи в категории Здоровье и красота
from io import StringIO

import pandas as pd
import requests

URL = "https://raw.githubusercontent.com/selva86/datasets/master/supermarket_sales.csv"
BEAUTY_LINE = "Health and beauty"

class SupermarketBeautySource(BaseSource):

    cache_name = "beauty_sales.csv"

    def fetch(self) -> pd.DataFrame:
        text = requests.get(URL, timeout=30, headers={"User-Agent": "Mozilla/5.0"}).text
        df = pd.read_csv(StringIO(text))
        beauty = df[df["Product line"] == BEAUTY_LINE].copy()
        out = pd.DataFrame(
            {
                "city": beauty["City"],
                "customer_type": beauty["Customer type"].map({"Member": "Постоянный", "Normal": "Обычный"}),
                "gender": beauty["Gender"].map({"Female": "Женский", "Male": "Мужской"}),
                "unit_price": beauty["Unit price"].astype(float),
                "quantity": beauty["Quantity"].astype(int),
                "total": beauty["Total"].astype(float),
                "rating": beauty["Rating"].astype(float),
            }
        )
        return out.reset_index(drop=True)

In [ ]:
#Курс рубля к доллару
from xml.etree import ElementTree as ET

import pandas as pd
import requests

CBR_URL = "https://www.cbr.ru/scripts/XML_daily.asp"
FX_URL = "https://open.er-api.com/v6/latest/USD"
FALLBACK_USD = 72.0
HEADERS = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"}

class CbrRateSource(BaseSource):

    cache_name = "cbr_usd.csv"

    def fetch(self) -> pd.DataFrame:
        for getter in (self._from_cbr, self._from_fx_api):
            try:
                return getter()
            except Exception:
                continue
        raise RuntimeError("ни один источник курса недоступен")

    def _from_cbr(self) -> pd.DataFrame:
        resp = requests.get(CBR_URL, timeout=6, headers=HEADERS)
        resp.raise_for_status()
        resp.encoding = "windows-1251"
        root = ET.fromstring(resp.text)
        date = root.attrib.get("Date")
        for valute in root.findall("Valute"):
            if valute.findtext("CharCode") == "USD":
                value = float(valute.findtext("Value").replace(",", ".").replace(" ", ""))
                nominal = float(valute.findtext("Nominal").replace(",", "."))
                return self._row(date, value / nominal, "ЦБ РФ")
        raise RuntimeError("в ответе ЦБ нет доллара")

    def _from_fx_api(self) -> pd.DataFrame:
        data = requests.get(FX_URL, timeout=20, headers=HEADERS).json()
        rate = float(data["rates"]["RUB"])
        return self._row(data.get("time_last_update_utc", ""), rate, "exchangerate-api")

    @staticmethod
    def _row(date, rate, source) -> pd.DataFrame:
        return pd.DataFrame([{"date": date, "char_code": "USD", "rate": round(rate, 4), "source": source}])

    @classmethod
    def usd_rate(cls, cache_dir) -> float:
        try:
            df = cls(cache_dir).load()
            return float(df["rate"].iloc[0])
        except Exception:
            return FALLBACK_USD

In [ ]:
#Выгрузка постов и комментариев из пабликов VK
import os
import time

import pandas as pd
import requests

DEFAULT_GROUPS = ["skincare_ru", "beautyhack", "stylebook", "myskincarediary"]

API = "https://api.vk.com/method/{method}"
API_VERSION = "5.199"

class VKParser(BaseSource):

    cache_name = "vk_posts.csv"

    def __init__(self, cache_dir, groups=None, posts_per_group=100, pause=0.35):
        super().__init__(cache_dir)
        self.groups = groups or DEFAULT_GROUPS
        self.posts_per_group = posts_per_group
        self.pause = pause
        self.token = os.getenv("VK_TOKEN")

    def fetch(self) -> pd.DataFrame:
        if not self.token:
            raise RuntimeError("нет VK_TOKEN")
        rows = []
        for group in self.groups:
            rows.extend(self._fetch_wall(group))
            time.sleep(self.pause)
        if not rows:
            raise RuntimeError("VK вернул пустую выдачу")
        return pd.DataFrame(rows)

    def _fetch_wall(self, domain: str) -> list:
        params = {
            "domain": domain,
            "count": min(self.posts_per_group, 100),
            "access_token": self.token,
            "v": API_VERSION,
        }
        resp = requests.get(API.format(method="wall.get"), params=params, timeout=15)
        resp.raise_for_status()
        payload = resp.json()
        if "error" in payload:
            raise RuntimeError(payload["error"].get("error_msg", "VK API error"))

        rows = []
        for item in payload.get("response", {}).get("items", []):
            text = (item.get("text") or "").strip()
            if not text:
                continue
            rows.append(
                {
                    "group": domain,
                    "text": text,
                    "likes": item.get("likes", {}).get("count", 0),
                    "comments": item.get("comments", {}).get("count", 0),
                    "reposts": item.get("reposts", {}).get("count", 0),
                }
            )
        return rows

## 3. Анализ

In [ ]:
from dataclasses import dataclass

import pandas as pd
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

FEATURES = ["age", "annual_income_kusd", "spending_score"]

@dataclass
class Persona:
    name: str
    share: float
    size: int
    avg_age: float
    avg_income: float
    avg_spending: float
    female_share: float
    summary: str = ""

    def as_dict(self) -> dict:
        return {
            "Портрет": self.name,
            "Доля, %": round(self.share * 100, 1),
            "Человек": self.size,
            "Возраст": round(self.avg_age, 1),
            "Доход, тыс. $/год": round(self.avg_income, 1),
            "Индекс трат": round(self.avg_spending, 1),
            "Женщин, %": round(self.female_share * 100, 1),
        }

class AudienceProfiler:
    def __init__(self, customers: pd.DataFrame, n_clusters: int = 4, seed: int = 42):
        self.customers = customers.copy()
        self.n_clusters = n_clusters
        self.seed = seed
        self._fitted = None

    def fit(self) -> pd.DataFrame:
        x = StandardScaler().fit_transform(self.customers[FEATURES])
        model = KMeans(n_clusters=self.n_clusters, random_state=self.seed, n_init=10)
        self.customers["cluster"] = model.fit_predict(x)
        self._fitted = self.customers
        return self.customers

    def build(self) -> list[Persona]:
        if self._fitted is None:
            self.fit()
        total = len(self.customers)
        personas = []
        for cluster, group in self.customers.groupby("cluster"):
            p = Persona(
                name=self._name_for(group),
                share=len(group) / total,
                size=len(group),
                avg_age=group["age"].mean(),
                avg_income=group["annual_income_kusd"].mean(),
                avg_spending=group["spending_score"].mean(),
                female_share=(group["gender"] == "Женский").mean(),
            )
            p.summary = self._describe(p)
            personas.append(p)
        return sorted(personas, key=lambda p: p.avg_spending, reverse=True)

    def _name_for(self, group: pd.DataFrame) -> str:
        income = group["annual_income_kusd"].mean()
        spend = group["spending_score"].mean()
        age = group["age"].mean()
        med_income = self.customers["annual_income_kusd"].median()
        med_spend = self.customers["spending_score"].median()

        high_income = income >= med_income
        high_spend = spend >= med_spend
        young = age < self.customers["age"].median()

        if high_spend and not high_income and young:
            return "Активная молодежь с небольшим доходом"
        if high_income and high_spend:
            return "Состоятельные и неэкономные"
        if high_income and not high_spend:
            return "Состоятельные, но экономные"
        if not high_income and not high_spend:
            return "Экономные"
        return "Умеренный сегмент"

    @staticmethod
    def _describe(p: Persona) -> str:
        return (
            f"{p.name}: примерно {p.size} человек ({p.share*100:.0f}% базы), средний "
            f"возраст {p.avg_age:.0f}, доход около {p.avg_income:.0f} тыс. $ в год, "
            f"индекс трат {p.avg_spending:.0f} из 100. Женщин {p.female_share*100:.0f}%."
        )

    def to_frame(self) -> pd.DataFrame:
        return pd.DataFrame([p.as_dict() for p in self.build()])

In [ ]:
from dataclasses import dataclass

import numpy as np
import pandas as pd
from scipy import stats

@dataclass
class PriceEstimate:
    mean_price: float
    median_price: float
    ci_low: float
    ci_high: float
    recommended_price: float
    coverage: float
    usd_to_rub: float

    def describe(self) -> str:
        return (
            f"Средняя цена за бьюти-товар {self.mean_price:.1f} $, медиана "
            f"{self.median_price:.1f} $. С вероятностью 95% настоящее среднее лежит "
            f"в диапазоне {self.ci_low:.1f}-{self.ci_high:.1f} $. Рекомендуемая цена "
            f"{self.recommended_price:.1f} $ (около {self.recommended_price*self.usd_to_rub:,.0f} руб "
            f"по курсу ЦБ {self.usd_to_rub:.2f}) по карману {self.coverage*100:.0f}% покупателей."
        ).replace(",", " ")

class WillingnessToPay:

    def __init__(self, beauty: pd.DataFrame, usd_to_rub: float = 95.0, price_col: str = "unit_price"):
        self.data = beauty
        self.usd_to_rub = usd_to_rub
        self.price_col = price_col

    def confidence_interval(self, confidence: float = 0.95) -> tuple[float, float]:
        x = self.data[self.price_col]
        n = len(x)
        mean = x.mean()
        se = x.std(ddof=1) / np.sqrt(n)
        margin = stats.t.ppf((1 + confidence) / 2, df=n - 1) * se
        return mean - margin, mean + margin

    def price_estimate(self, percentile: int = 60) -> PriceEstimate:
        x = self.data[self.price_col]
        ci_low, ci_high = self.confidence_interval()
        price = round(float(np.percentile(x, percentile)), 1)
        coverage = (x >= price).mean()
        return PriceEstimate(
            mean_price=x.mean(),
            median_price=x.median(),
            ci_low=ci_low,
            ci_high=ci_high,
            recommended_price=price,
            coverage=coverage,
            usd_to_rub=self.usd_to_rub,
        )

    def to_rub(self, usd: float) -> float:
        return usd * self.usd_to_rub

    def by_group(self, col: str) -> pd.DataFrame:
        agg = self.data.groupby(col)[self.price_col].agg(["count", "mean", "median", "std"]).round(2)
        agg.columns = ["Покупок", "Средняя цена", "Медиана", "Разброс"]
        return agg.sort_values("Средняя цена", ascending=False).reset_index().rename(columns={col: "Группа"})

In [ ]:
#Боли аудитории
import pandas as pd

class PainAnalyzer:
    def __init__(self, pain_keywords: dict[str, list[str]]):
        self.pain_keywords = pain_keywords
    def from_texts(self, posts: pd.DataFrame, text_col: str = "text") -> pd.DataFrame:
        tally = {pain: 0 for pain in self.pain_keywords}
        for raw in posts[text_col].dropna():
            text = str(raw).lower()
            for pain, markers in self.pain_keywords.items():
                if any(marker in text for marker in markers):
                    tally[pain] += 1
        series = pd.Series(tally).sort_values(ascending=False)
        total = max(series.sum(), 1)
        return pd.DataFrame(
            {"Боль": series.index, "Упоминаний": series.values, "Доля, %": (series / total * 100).round(1).values}
        )

In [ ]:
from dataclasses import dataclass

import numpy as np
import pandas as pd
from scipy import stats
ALPHA = 0.05
@dataclass
class TestResult:
    name: str
    h0: str
    h1: str
    test: str
    statistic: float
    p_value: float
    reject_h0: bool
    verdict: str
    def as_dict(self) -> dict:
        return {
            "Гипотеза": self.name,
            "Критерий": self.test,
            "Статистика": round(self.statistic, 3),
            "p-value": round(self.p_value, 4),
            "Отвергаем H0": "да" if self.reject_h0 else "нет",
            "Вывод": self.verdict,
        }

class HypothesisTester:

    def __init__(self, customers: pd.DataFrame, beauty: pd.DataFrame, alpha: float = ALPHA):
        self.customers = customers
        self.beauty = beauty
        self.alpha = alpha

    def run_all(self) -> pd.DataFrame:
        results = [
            self.income_vs_spending(),
            self.members_pay_more(),
            self.gender_price_difference(),
            self.young_spend_more(),
        ]
        return pd.DataFrame([r.as_dict() for r in results])

    def income_vs_spending(self) -> TestResult:
        rho, p = stats.spearmanr(self.customers["annual_income_kusd"], self.customers["spending_score"])
        reject = p < self.alpha
        verdict = f"Связь есть, коэффициент {rho:.2f}" if reject else "Значимой связи не обнаружено"
        return TestResult(
            "Доход связан с размером трат",
            "Доход и размер трат независимы",
            "Между доходом и размером трат есть связь",
            "Корреляция Спирмена",
            rho, p, reject, verdict,
        )

    def members_pay_more(self) -> TestResult:
        member = self.beauty.query("customer_type == 'Постоянный'")["unit_price"]
        normal = self.beauty.query("customer_type == 'Обычный'")["unit_price"]
        stat, p = stats.mannwhitneyu(member, normal, alternative="greater")
        reject = p < self.alpha
        verdict = "Постоянные платят значимо больше" if reject else "Значимой разницы не нашли"
        return TestResult(
            "Постоянные платят больше обычных",
            "Цены у постоянных и обычных одинаковы",
            "У постоянных покупателей цена выше",
            "Манна-Уитни (односторонний)",
            stat, p, reject, verdict,
        )

    def gender_price_difference(self) -> TestResult:
        female = self.beauty.query("gender == 'Женский'")["unit_price"]
        male = self.beauty.query("gender == 'Мужской'")["unit_price"]
        stat, p = stats.ttest_ind(female, male, equal_var=False)
        reject = p < self.alpha
        verdict = "Цена различается по полу значимо" if reject else "Значимой разницы по полу не нашли"
        return TestResult(
            "Цена бьюти-покупки различается по полу",
            "Средняя цена у мужчин и женщин одинакова",
            "Средняя цена у мужчин и женщин различается",
            "t-критерий Уэлча (двусторонний)",
            stat, p, reject, verdict,
        )
    def young_spend_more(self) -> TestResult:
        med_age = self.customers["age"].median()
        young = self.customers[self.customers["age"] < med_age]["spending_score"]
        older = self.customers[self.customers["age"] >= med_age]["spending_score"]
        stat, p = stats.mannwhitneyu(young, older, alternative="greater")
        reject = p < self.alpha
        verdict = "Молодежь тратит значительно активнее" if reject else "Значимой разницы по возрасту не нашли"
        return TestResult(
            "Молодежь тратит активнее старшего поколения",
            "Индекс трат у молодого и старшего поколения одинаковый",
            "У молодых размер трат выше",
            "Манна-Уитни (односторонний)",
            stat, p, reject, verdict,
        )

    def sample_quality(self) -> dict:
        x = self.beauty["unit_price"]
        shapiro_stat, shapiro_p = stats.shapiro(x.sample(min(len(x), 500), random_state=0))
        z = stats.norm.ppf(0.975)
        margin = 0.05 * x.mean()
        needed_n = int(np.ceil((z * x.std(ddof=1) / margin) ** 2))
        return {
            "Размер выборки (бьюти-покупки)": len(x),
            "Шапиро-Уилк p-value": round(shapiro_p, 4),
            "Распределение нормальное": "нет" if shapiro_p < self.alpha else "да",
            "Нужный объем для точности 5%": needed_n,
            "Объема достаточно": "да" if len(x) >= needed_n else "нет",
        }

In [ ]:
#Оценка размера рынка
from dataclasses import dataclass
import pandas as pd

def human(n: float) -> str:
    if abs(n) >= 1_000_000:
        return f"{n / 1_000_000:.2f} млн"
    if abs(n) >= 1_000:
        return f"{n / 1_000:.2f} тыс."
    return f"{n:.0f}"

@dataclass
class MarketSize:
    tam: int
    sam: int
    som: int
    arpu: float
    som_revenue: float
    assumptions: dict

    def describe(self) -> str:
        return (
            f"TAM около {human(self.tam)} человек (молодежь крупных городов в онлайне). "
            f"SAM около {human(self.sam)} человек (интересна тема внешности). "
            f"SOM на старте около {human(self.som)} человек. При среднем чеке "
            f"{self.arpu:,.0f} руб это примерно {human(self.som_revenue)} руб выручки."
        ).replace(",", " ")

class MarketSizer:
    def __init__(self, cities: pd.DataFrame, wb_latest: dict, beauty: pd.DataFrame, usd_to_rub: float = 95.0):
        self.cities = cities
        self.wb = wb_latest
        self.beauty = beauty
        self.usd_to_rub = usd_to_rub

    def estimate(
        self,
        top_n_cities: int = 30,
        youth_share: float = 0.18,
        interest_share: float = 0.40,
        capture_rate: float = 0.02,
    ) -> MarketSize:
        city_pop = int(self.cities.head(top_n_cities)["population"].sum())
        online_share = self.wb.get("Интернет-пользователи, %", {}).get("value", 90.0) / 100
        arpu = float(self.beauty["unit_price"].mean()) * self.usd_to_rub
        tam = int(city_pop * youth_share * online_share)
        sam = int(tam * interest_share)
        som = int(sam * capture_rate)

        return MarketSize(
            tam=tam,
            sam=sam,
            som=som,
            arpu=arpu,
            som_revenue=som * arpu,
            assumptions={
                "Городов учтено": top_n_cities,
                "Население этих городов": city_pop,
                "Доля молодежи 16-29": youth_share,
                "Доля онлайн (по данным World Bank)": round(online_share, 3),
                "Доля с интересом к теме": interest_share,
                "Охват на старте": capture_rate,
                "Курс доллара ЦБ": round(self.usd_to_rub, 2),
            },
        )

    def funnel_frame(self, size: MarketSize) -> pd.DataFrame:
        people = [size.tam, size.sam, size.som]
        return pd.DataFrame(
            {
                "Этап": ["TAM (вся молодёжь онлайн)", "SAM (интересна эта тема)", "SOM (охват на старте)"],
                "Человек": people,
                "Подпись": [human(p) for p in people],
            }
        )

## 4. Графики и таблицы

In [ ]:
import sys
from pathlib import Path
import matplotlib.pyplot as plt
import pandas as pd

HEADER_COLOR = TABLE_HEADER
ROW_ALT = TABLE_ROW_ALT
TEXT_DARK = "#222222"

class TableRenderer:

    def __init__(self, figures_dir: Path):
        self.dir = Path(figures_dir)
        self.dir.mkdir(parents=True, exist_ok=True)
    def render(self, df: pd.DataFrame, name: str, title: str = "", col_width=None) -> Path:
        df = df.copy()
        for col in df.columns:
            if pd.api.types.is_numeric_dtype(df[col]):
                df[col] = df[col].map(self._fmt_number)

        n_rows, n_cols = df.shape
        fig_w = col_width or max(7, 1.7 * n_cols)
        fig, ax = plt.subplots(figsize=(fig_w, 0.5 * n_rows + 1.4))
        ax.axis("off")
        if title:
            ax.set_title(title, fontsize=14, fontweight="bold", pad=16, color=TEXT_DARK)

        table = ax.table(
            cellText=df.values,
            colLabels=df.columns,
            cellLoc="center",
            loc="center",
        )
        table.auto_set_font_size(False)
        table.set_fontsize(10)
        table.scale(1, 1.5)

        for (row, _), cell in table.get_celld().items():
            cell.set_edgecolor("white")
            if row == 0:
                cell.set_facecolor(HEADER_COLOR)
                cell.set_text_props(color="white", fontweight="bold")
            else:
                cell.set_facecolor(ROW_ALT if row % 2 == 0 else "white")
                cell.set_text_props(color=TEXT_DARK)

        table.auto_set_column_width(col=list(range(n_cols)))
        path = self.dir / name
        fig.tight_layout()
        fig.savefig(path, dpi=150, bbox_inches="tight")
        plt.close(fig)
        print(f"таблица сохранена: {name}")
        return path

    @staticmethod
    def _fmt_number(x):
        if pd.isna(x):
            return ""
        if isinstance(x, float) and not x.is_integer():
            return f"{x:,.2f}".replace(",", " ")
        return f"{int(x):,}".replace(",", " ")

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", font_scale=1.05)

class Plotter:
    def __init__(self, figures_dir: Path):
        self.dir = Path(figures_dir)
        self.dir.mkdir(parents=True, exist_ok=True)

    def _save(self, fig, name: str) -> Path:
        path = self.dir / name
        fig.tight_layout()
        fig.savefig(path, dpi=150, bbox_inches="tight")
        plt.close(fig)
        print(f"график сохранён: {name}")
        return path

    def segments_scatter(self, customers):
        fig, ax = plt.subplots(figsize=(8, 5))
        sns.scatterplot(
            data=customers, x="annual_income_kusd", y="spending_score",
            hue="cluster", palette=PALETTE, s=60, ax=ax,
        )
        ax.set_title("Сегменты покупателей: доход и активность трат")
        ax.set_xlabel("Годовой доход, тыс. $")
        ax.set_ylabel("Индекс трат (1-100)")
        ax.legend(title="Сегмент")
        return self._save(fig, "01_segments_scatter.png")

    def age_distribution(self, customers):
        fig, ax = plt.subplots(figsize=(8, 5))
        sns.histplot(data=customers, x="age", hue="gender", multiple="stack", bins=14, palette=GENDER_COLORS, ax=ax)
        ax.set_title("Распределение возраста покупателей")
        ax.set_xlabel("Возраст")
        ax.set_ylabel("Количество")
        return self._save(fig, "02_age_distribution.png")

    def spending_by_segment(self, customers):
        fig, ax = plt.subplots(figsize=(8, 5))
        order = customers.groupby("cluster")["spending_score"].median().sort_values(ascending=False).index
        sns.boxplot(data=customers, x="cluster", y="spending_score", order=order, color=ACCENT_COLOR, ax=ax)
        ax.set_title("Активность трат по сегментам")
        ax.set_xlabel("Сегмент (кластер)")
        ax.set_ylabel("Индекс трат")
        return self._save(fig, "03_spending_by_segment.png")

    def price_distribution(self, beauty):
        fig, ax = plt.subplots(figsize=(8, 5))
        sns.histplot(data=beauty, x="unit_price", bins=20, color=ACCENT_COLOR, ax=ax)
        ax.set_title("Цены на бьюти-товары (реальные продажи)")
        ax.set_xlabel("Цена за товар, $")
        ax.set_ylabel("Количество покупок")
        return self._save(fig, "04_price_distribution.png")

    def price_by_group(self, beauty):
        fig, ax = plt.subplots(figsize=(8, 5))
        sns.boxplot(data=beauty, x="customer_type", y="unit_price", hue="gender", palette=GENDER_COLORS, ax=ax)
        ax.set_title("Цена покупки по типу клиента и полу")
        ax.set_xlabel("")
        ax.set_ylabel("Цена за товар, $")
        return self._save(fig, "05_price_by_group.png")

    def top_pains(self, pains_df):
        fig, ax = plt.subplots(figsize=(8, 5))
        col = pains_df.columns[-1]
        data = pains_df.sort_values(col, ascending=True)
        sns.barplot(data=data, y="Боль", x=col, color=ACCENT_COLOR, ax=ax)
        ax.set_title("Главные боли аудитории (соцсети)")
        ax.set_xlabel("Доля упоминаний, %")
        ax.set_ylabel("")
        return self._save(fig, "06_top_pains.png")

    def top_cities(self, cities):
        fig, ax = plt.subplots(figsize=(8, 5))
        data = cities.head(12).iloc[::-1]
        sns.barplot(x=data["population"] / 1000, y=data["city"], color=ACCENT_COLOR, ax=ax)
        ax.set_title("Крупнейшие города России по населению (Wikipedia)")
        ax.set_xlabel("Население, тыс. человек")
        ax.set_ylabel("")
        return self._save(fig, "07_top_cities.png")

    def internet_trend(self, wb):
        fig, ax = plt.subplots(figsize=(8, 5))
        data = wb[wb["indicator"] == "Интернет-пользователи, %"].sort_values("year")
        sns.lineplot(data=data, x="year", y="value", marker="o", color=ACCENT_COLOR, ax=ax)
        ax.set_title("Охват интернетом в России (World Bank)")
        ax.set_xlabel("Год")
        ax.set_ylabel("Доля населения, %")
        return self._save(fig, "08_internet_trend.png")

    def market_funnel(self, funnel_df):
        fig, ax = plt.subplots(figsize=(8, 5))
        sns.barplot(data=funnel_df, x="Человек", y="Этап", color=ACCENT_COLOR, ax=ax)
        labels = funnel_df["Подпись"] if "Подпись" in funnel_df.columns else funnel_df["Человек"]
        for i, (v, label) in enumerate(zip(funnel_df["Человек"], labels)):
            ax.text(v, i, f"  {label}", va="center", fontsize=10)
        ax.set_title("Воронка рынка: TAM / SAM / SOM")
        ax.set_xlabel("Человек")
        ax.set_ylabel("")
        return self._save(fig, "09_market_funnel.png")

    def build_all(self, customers, beauty, pains_df):
        return [
            self.segments_scatter(customers),
            self.age_distribution(customers),
            self.spending_by_segment(customers),
            self.price_distribution(beauty),
            self.price_by_group(beauty),
            self.top_pains(pains_df),
        ]

## 5. Сбор данных

Wikipedia, World Bank, датасеты и курс тянутся вживую. Посты VK задаём примером, чтобы блокнот работал без токена.

In [ ]:
customers = MallCustomersSource(DATA_RAW).load()
beauty = SupermarketBeautySource(DATA_RAW).load()
cities = WikipediaCitiesParser(DATA_RAW).load()
wb = WorldBankSource(DATA_RAW).load()
wb_latest = WorldBankSource.latest(wb)
usd_rate = CbrRateSource.usd_rate(DATA_RAW)
print('Курс доллара:', usd_rate)

_vk_samples = [
    'Совсем не знаю с чего начать уход за кожей, слишком много информации, глаза разбегаются',
    'Хочу нормальный гардероб, но не понимаю что мне идёт и не мой стиль вообще',
    'Ходить к стилисту дорого, цена кусается, а выглядеть хочется хорошо',
    'Стесняюсь своей внешности, комплексую перед собеседованиями',
    'Нет времени разбираться во всех этих сыворотках, времени не хватает',
    'Боюсь испортить кожу новыми средствами, страшно пробовать наугад',
    'Косметолог дорого, не по карману студенту, ищу что подешевле',
    'Запуталась в порядке нанесения средств, не знаю что за чем',
    'Хочу подтянуть внешний вид перед стажировкой, но не нравлюсь себе',
    'Купила кучу косметики и всё равно не понимаю свой тип кожи',
]
posts = pd.DataFrame({'text': _vk_samples * 22})
customers.head()

## 6. Размер рынка

In [ ]:
sizer = MarketSizer(cities, wb_latest, beauty, usd_to_rub=usd_rate)
size = sizer.estimate()
print(size.describe())
size.assumptions

## 7. Портреты ЦА (кластеризация KMeans)

In [ ]:
profiler = AudienceProfiler(customers)
profiler.fit()
for p in profiler.build():
    print('-', p.summary)
profiler.to_frame()

## 8. Платёжеспособность

In [ ]:
wtp = WillingnessToPay(beauty, usd_to_rub=usd_rate)
print(wtp.price_estimate().describe())
wtp.by_group('customer_type')

## 9. Боли аудитории

In [ ]:
pains = PainAnalyzer(PAIN_KEYWORDS).from_texts(posts)
pains

## 10. Проверка гипотез и оценка выборки

In [ ]:
tester = HypothesisTester(profiler.customers, beauty)
display(tester.run_all())
tester.sample_quality()

## 11. Графики

In [ ]:
import matplotlib.pyplot as plt
from IPython.display import Image, display
plotter = Plotter(FIGURES)
sizer = MarketSizer(cities, wb_latest, beauty, usd_to_rub=usd_rate)
funnel = sizer.funnel_frame(size)
paths = [
    plotter.market_funnel(funnel), plotter.top_cities(cities), plotter.internet_trend(wb),
    *plotter.build_all(profiler.customers, beauty, pains),
]
for pth in paths:
    display(Image(str(pth)))

## 12. Дэшборд (Dash)

Ячейка ниже собирает интерактивный дэшборд. Запуск в блокноте: снять комментарий с последней строки (откроется внутри ноутбука).

In [ ]:
#Открывается через запуск: python app/dashboard.py и http://127.0.0.1:8050

import sys
from pathlib import Path

import plotly.express as px
from dash import Dash, dash_table, dcc, html
from dash.dependencies import Input, Output

CUSTOMERS = AudienceProfiler(MallCustomersSource(DATA_RAW).load()).fit()
BEAUTY = SupermarketBeautySource(DATA_RAW).load()
POSTS = VKParser(DATA_RAW).load()
CITIES = WikipediaCitiesParser(DATA_RAW).load()
WB = WorldBankSource(DATA_RAW).load()
WB_LATEST = WorldBankSource.latest(WB)
USD_RATE = CbrRateSource.usd_rate(DATA_RAW)
MARKET = MarketSizer(CITIES, WB_LATEST, BEAUTY, usd_to_rub=USD_RATE).estimate()
PAINS = PainAnalyzer(PAIN_KEYWORDS).from_texts(POSTS)
HYP = HypothesisTester(CUSTOMERS, BEAUTY).run_all()
COLORWAY = SEQ_COLORS

app = Dash(__name__, title="Анализ ЦА")
server = app.server

def card(title, value):
    return html.Div(
        style={"flex": "1", "background": TABLE_ROW_ALT, "borderRadius": "12px", "padding": "16px"},
        children=[html.Div(title, style={"fontSize": "13px", "color": "#555"}),
                  html.Div(value, style={"fontSize": "24px", "fontWeight": "bold"})],
    )

def market_funnel_fig():
    funnel = MarketSizer(CITIES, WB_LATEST, BEAUTY, usd_to_rub=USD_RATE).funnel_frame(MARKET)
    fig = px.funnel(funnel, x="Человек", y="Этап", text="Подпись", title="Воронка рынка: TAM / SAM / SOM",
                    color_discrete_sequence=[ACCENT_COLOR])
    return fig

def top_cities_fig():
    data = CITIES.head(12)
    fig = px.bar(data, x="population", y="city", orientation="h",
                 title=f"Крупнейшие города России, {int(CITIES['year'].iloc[0])} (Wikipedia)",
                 color_discrete_sequence=[ACCENT_COLOR])
    fig.update_layout(yaxis={"categoryorder": "total ascending"}, xaxis_title="Население", yaxis_title="")
    return fig

def internet_fig():
    data = WB[WB["indicator"] == "Интернет-пользователи, %"].sort_values("year")
    fig = px.line(data, x="year", y="value", markers=True,
                  title="Охват интернетом в России (World Bank)", color_discrete_sequence=[ACCENT_COLOR])
    fig.update_layout(xaxis_title="Год", yaxis_title="Доля населения, %")
    return fig

def pains_fig():
    return px.bar(PAINS.sort_values("Доля, %"), x="Доля, %", y="Боль", orientation="h",
                  title="Главные боли аудитории (соцсети)", color_discrete_sequence=[ACCENT_COLOR])

app.layout = html.Div(
    style={"maxWidth": "1100px", "margin": "0 auto", "fontFamily": "Inter, Arial, sans-serif", "padding": "24px"},
    children=[
        html.H1("Целевая аудитория: онлайн-курсы по внешнему виду"),
        html.P("Кому продаём, какие боли клиента и сколько они готовы платить. Данные спарсены из пабликов ВК и с сайта hh.ru."),
        html.Div(
            style={"display": "flex", "gap": "12px", "alignItems": "center", "margin": "16px 0"},
            children=[
                html.Label("Пол:"),
                dcc.Dropdown(
                    id="gender",
                    options=[{"label": g, "value": g} for g in ["Все", "Женский", "Мужской"]],
                    value="Все", clearable=False, style={"width": "260px"},
                ),
            ],
        ),
        html.Div(id="cards", style={"display": "flex", "gap": "16px", "marginBottom": "8px"}),
        dcc.Tabs(
            [
                dcc.Tab(label="Рынок", children=[
                    html.Br(),
                    html.P("Размер рынка: население городов из Wikipedia, охват интернетом из World Bank, "
                           "средний чек из реальных продаж бьюти-товаров."),
                    dcc.Graph(figure=market_funnel_fig()),
                    dcc.Graph(figure=top_cities_fig()),
                    dcc.Graph(figure=internet_fig()),
                ]),
                dcc.Tab(label="Сегменты и портреты", children=[
                    dcc.Graph(id="segments"),
                    dcc.Graph(id="age"),
                ]),
                dcc.Tab(label="Платёжеспособность", children=[
                    dcc.Graph(id="price_dist"),
                    dcc.Graph(id="price_group"),
                ]),
                dcc.Tab(label="Боли клиента", children=[dcc.Graph(figure=pains_fig())]),
                dcc.Tab(label="Гипотезы", children=[
                    html.Br(),
                    dash_table.DataTable(
                        data=HYP.to_dict("records"),
                        columns=[{"name": c, "id": c} for c in HYP.columns],
                        style_cell={"textAlign": "left", "fontFamily": "Arial", "padding": "8px", "whiteSpace": "normal"},
                        style_header={"fontWeight": "bold", "backgroundColor": TABLE_ROW_ALT},
                    ),
                ]),
            ]
        ),
    ],
)


def filtered_customers(gender):
    return CUSTOMERS if gender == "Все" else CUSTOMERS[CUSTOMERS["gender"] == gender]

def filtered_beauty(gender):
    return BEAUTY if gender == "Все" else BEAUTY[BEAUTY["gender"] == gender]

@app.callback(
    Output("cards", "children"),
    Output("segments", "figure"),
    Output("age", "figure"),
    Output("price_dist", "figure"),
    Output("price_group", "figure"),
    Input("gender", "value"),
)
def update(gender):
    cust = filtered_customers(gender)
    beauty = filtered_beauty(gender)
    estimate = WillingnessToPay(beauty, usd_to_rub=USD_RATE).price_estimate() if len(beauty) > 5 else None

    cards = [
        card("Покупателей в базе", f"{len(cust)}"),
        card("Средняя цена", f"{beauty['unit_price'].mean():.1f} $ / {beauty['unit_price'].mean()*USD_RATE:.0f} руб"),
        card("Оптимальная цена", f"{estimate.recommended_price:.1f} $ / {estimate.recommended_price*USD_RATE:.0f} руб" if estimate else "-"),
        card("Курс ЦБ", f"{USD_RATE:.2f} руб/$"),
    ]

    seg_fig = px.scatter(cust, x="annual_income_kusd", y="spending_score", color="cluster",
                         title="Сегменты: доход и активность трат", color_continuous_scale=COLORWAY,
                         labels={"annual_income_kusd": "Доход, тыс. $/год", "spending_score": "Индекс трат"})
    age_fig = px.histogram(cust, x="age", color="gender", nbins=14, title="Возраст покупателей",
                           color_discrete_map=GENDER_COLORS)
    age_fig.update_layout(bargap=0.05)
    price_dist = px.histogram(beauty, x="unit_price", nbins=20, title="Цены на бьюти-товары",
                              color_discrete_sequence=[ACCENT_COLOR])
    price_dist.update_layout(xaxis_title="Цена за товар, $", yaxis_title="Покупок")
    price_group = px.box(beauty, x="customer_type", y="unit_price", color="gender",
                         title="Цена покупки по типу клиента", color_discrete_map=GENDER_COLORS)
    price_group.update_layout(xaxis_title="", yaxis_title="Цена за товар, $")
    return cards, seg_fig, age_fig, price_dist, price_group